# Lab 04｜旧任务中的纠正，怎样进入新任务？

这是专栏《自进化 Agent 实战》04 讲《AGENTS.md：如何设计规则指导 Agent 未来行动》的配套练习。练习使用构造的事故调查材料，目标是掌握处理真实场景的能力。

## 题面

团队已把“时间上的先后不能直接写成因果”确定为所有事故报告都要遵守的要求，支付服务另有一条本地排查要求。你发现从项目根目录启动的调查能收到通用要求，却收不到支付服务的要求；从服务目录启动时两条都能收到。现在还要临时替换支付服务的本地要求，同时保留项目通用要求。你会怎样安排规则文件和启动位置？同一目录放了两份规则时，应该怎样选取？又怎样证明新任务收到的是预期组合，而不是恰好沿用了上一次对话里的纠正？

## 参考答案

我会先分清一次对话里的纠正，和项目以后都要遵守的要求。旧任务里改过报告，不代表新任务自动得到那句话；把旧事故整段复制过去，还会把无关的结论一起带进来。这里团队已经确认了稳定要求，就应该把通用要求和服务要求分别放在对应目录，让程序在新任务开始时自己加载，而不是靠上一次对话残留。

这套查找不是递归扫描整个项目。它先从当前工作目录向上找项目根标记，再收集根目录到当前目录之间的规则，最后按从根到当前的顺序组合。所以从根目录启动时，不会向下发现服务子目录里的规则；从服务目录启动时，根规则先进入输入，服务规则随后补充。启动目录本身就是查找范围的一部分，不能只在对话里说“遵守服务规则”，就假定文件已经加载。

同一层目录最多采用一份规则。临时替换服务要求时，我会在服务目录放一份优先文件，并把这一层仍要保留的内容写完整；它替换的是这一层的整份规则，不会自动继承同目录另一份文件的其他内容。项目根目录的规则属于另一层，仍然照常加载。

验证时看新任务的实际输入，不以回答碰巧正确为依据。先把纠正留在旧任务里，再开新任务，确认它没有自动带过来；随后在根和服务目录放规则，用同一事故分别从两个目录启动，核对两级内容和先后位置；最后加入服务目录的优先文件，检查根规则仍在、优先文件已加入、同层原规则不再加载，再撤下临时文件。这些规则没有改模型参数，也不构成写入权限；输入里出现规则，只证明模型收到了要求，报告是否遵守还要另看。


## 准备

使用 Python 3.12 内核。需要 `GLM_API_KEY`，未设置时会弹出隐藏输入框；依赖自动补装。`lab.codex` 是真实 `Codex` 客户端。规则由 app-server 在创建会话时发现；下面直接改变 `thread_start(cwd=...)`，比较根目录、服务目录及优先规则文件的实际加载结果。


In [ ]:
import sys
from pathlib import Path
examples = next(p for base in (Path.cwd(), *Path.cwd().parents)
                for p in (base, base / "examples") if (p / "notebook04_setup.py").is_file())
sys.path.insert(0, str(examples))
from notebook04_setup import prepare
story = prepare()


## 上一次已经改对，为什么新的任务又错了？

这里起作用的是框架的新会话状态与规则重新加载：旧会话中的纠正，不会自动成为新会话的要求。
以两份构造事故作对照：旧初答按“先发生即原因”的旧稿写法生成，再纠正；另开会话调查新材料。
跑完对照“阶段/本次回答”，并检查新任务的实际输入；新回答即使谨慎，也不能证明旧纠正已经留下。


In [ ]:
from openai_codex import ApprovalMode, Codex, Sandbox
from notebook04_story import request_input
lab, work, payment = story.lab, story.work, story.payment
codex: Codex = lab.codex if lab else None
old_start = len(lab.requests) if lab else 0
old_thread = codex.thread_start(cwd=str(payment), sandbox=Sandbox.full_access,
    approval_mode=ApprovalMode.deny_all) if codex else None
old = corrected = None
if lab:
    turn = old_thread.turn(story.old_prompt)
    with lab.watch(turn):
        old = turn.run()
    lab.record_result(turn, old, show_result=False)
corrected_start = len(lab.requests) if lab else 0
if lab:
    turn = old_thread.turn(story.correction + "请据此修正刚才的结论，不调用工具。")
    with lab.watch(turn):
        corrected = turn.run()
    lab.record_result(turn, corrected, show_result=False)
new_start = len(lab.requests) if lab else 0
fresh_thread = codex.thread_start(cwd=str(payment), sandbox=Sandbox.full_access,
    approval_mode=ApprovalMode.deny_all) if codex else None


In [ ]:
fresh = None
if lab:
    turn = fresh_thread.turn(story.new_prompt)
    with lab.watch(turn):
        fresh = turn.run()
    lab.record_result(turn, fresh, show_result=False)
fresh_requests = lab.requests[new_start:] if lab else []
fresh_inputs = [request_input(request) for request in fresh_requests]
print("新任务输入中的旧纠正：", "存在" if any(story.correction in text for text in fresh_inputs) else "没有自动带入" if lab else "离线无请求")
story.show_history(old, corrected, fresh, old_thread, fresh_thread, (old_start, corrected_start, new_start))


## 哪些纠正值得留下？

这里起作用的是框架的项目规则加载；哪些纠正值得长期保留，仍由团队先确认，不能把整段旧记录当作规则。
从旧事故中分出实例名和连接池参数，再核对可复用的三条调查要求；这些实例与数值均为构造值。
跑完对照“留在旧任务/进入项目要求”；此时只阅读候选内容，还没有写入实验项目的规则文件。


In [ ]:
old_material = (work / "old-incident.md").read_text(encoding="utf-8")
task_specific = [line for line in old_material.splitlines() if "server-17" in line]
rule_text = (story.materials / "AGENTS.md").read_text(encoding="utf-8")
stable_requirements = [line.removeprefix("- ") for line in rule_text.splitlines() if line.startswith("- ")]


In [ ]:
local_text = (story.materials / "payment-service/AGENTS.md").read_text(encoding="utf-8")
print("拟保留的通用要求：\n" + rule_text)
print("服务本地要求另放一层：\n" + local_text)
story.show_selection(task_specific, stable_requirements, rule_text)


## 规则写在哪里，决定哪些任务会使用它

这里起作用的是框架的项目规则加载：从 `.git` 标记的根目录到当前工作目录（cwd）查找、组合，新会话开始时重新读取。
用同一份构造事故分别从根目录和 `payment-service/` 启动，检查两级范围与根规则在前的顺序。
再观察同层候选文件优先：`AGENTS.override.md` 替换服务这一层整份规则；核对根规则仍在、原服务规则未合并，随后撤下临时文件。


In [ ]:
(work / "AGENTS.md").write_text(rule_text, encoding="utf-8")
(payment / "AGENTS.md").write_text(local_text, encoding="utf-8")
root_start = len(lab.requests) if lab else 0
root_thread = codex.thread_start(cwd=str(work), sandbox=Sandbox.full_access,
    approval_mode=ApprovalMode.deny_all) if codex else None
root_result = local_result = None
if lab:
    turn = root_thread.turn(story.new_prompt)
    with lab.watch(turn):
        root_result = turn.run()
    lab.record_result(turn, root_result, show_result=False)
local_start = len(lab.requests) if lab else 0
local_thread = codex.thread_start(cwd=str(payment), sandbox=Sandbox.full_access,
    approval_mode=ApprovalMode.deny_all) if codex else None
if lab:
    turn = local_thread.turn(story.new_prompt)
    with lab.watch(turn):
        local_result = turn.run()
    lab.record_result(turn, local_result, show_result=False)
local_input = request_input(lab.requests[local_start]) if lab else ""
print("服务目录请求中两级规则的位置：", (local_input.find(story.root_rule), local_input.find(story.local_rule)) if lab else "离线无请求")
story.record_scope(root_result, local_result, root_thread, local_thread, root_start, local_start)


In [ ]:
override_start = len(lab.requests) if lab else 0
try:
    story.override_path.write_text(story.override_rule + "\n", encoding="utf-8")
    override_thread = codex.thread_start(cwd=str(payment), sandbox=Sandbox.full_access,
        approval_mode=ApprovalMode.deny_all) if codex else None
    override_result = None
    if lab:
        turn = override_thread.turn(story.new_prompt)
        with lab.watch(turn):
            override_result = turn.run()
        lab.record_result(turn, override_result, show_result=False)
    override_input = request_input(lab.requests[override_start]) if lab else ""
    print("优先文件实际进入输入：", story.override_rule in override_input if lab else "离线无请求")
finally:
    story.override_path.unlink(missing_ok=True)
story.show_rules(override_result, override_thread, override_start)
